# IPO Listing-Day Gain Prediction – Data Validation, Cleaning & Feature Engineering

**Course:** Introduction to Machine Learning (PCCAIML502)  
**Objective:** Prepare a clean, leakage-free baseline dataset for predicting Day-1 IPO listing gains (`target_open_gain_pct`, `target_typical_gain_pct`, `target_close_gain_pct`) across Indian Mainboard and SME IPOs (Nov 2023 – Nov 2026).

In this notebook, we walk through the compiled dataset field by field – inspecting for malformed values, withdrawn issues, unit/sign anomalies, and missingness – before engineering scale-invariant financial ratios, sentiment metrics, macro sector groupings, and log-transformed demand features.

In [1]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

RAW_PATH = "raw_ipo_dataset.csv"
CLEAN_PATH = "ipo_listing_gain_clean.csv"

exclude_on_load = {"target_hold_90d_alpha", "subs_source", "source_url"}
df_raw = pd.read_csv(
    RAW_PATH,
    dtype={"bse_code": str},
    usecols=lambda c: c not in exclude_on_load,
)
print(f"Loaded dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")

df = df_raw.copy()

Loaded dataset: 1064 rows, 75 columns


---
## Step 1: Duplicate & Identifier Integrity Check
Check whether any company or ISIN appears more than once, and inspect basic categorical distributions (`issue_type`, `exchange`, `is_sme`).

In [2]:
print("Duplicate company_name :", df["company_name"].duplicated().sum())
print("Duplicate non-null ISIN:", df["isin"].dropna().duplicated().sum())

# Normalize NSE SME exchange label
df.loc[(df["is_sme"] == 1) & (df["exchange"] == "NSE_ONLY"), "exchange"] = "NSE_EMERGE"

print("\nIssue Type distribution:")
print(df["issue_type"].value_counts(dropna=False))

print("\nExchange distribution:")
print(df["exchange"].value_counts(dropna=False))

print("\nSME vs Mainboard:")
print(df["is_sme"].value_counts(dropna=False))

Duplicate company_name : 0
Duplicate non-null ISIN: 0

Issue Type distribution:
issue_type
Book_Built     881
Fixed_Price    183
Name: count, dtype: int64

Exchange distribution:
exchange
NSE_EMERGE    381
BSE_SME       359
NSE_BSE       321
BSE_ONLY        3
Name: count, dtype: int64

SME vs Mainboard:
is_sme
1    740
0    324
Name: count, dtype: int64


---
## Step 2: Target & Listing Status Validation (Removing Unlisted / Withdrawn Issues)
Any IPO that was withdrawn, cancelled, or never listed cannot have a valid Day-1 listing price or gain target. Rows with missing `listing_date`, `issue_price`, or `target_*_gain_pct` are inspected and dropped.

In [3]:
malformed_listing = df[
    df["listing_date"].isna()
    | df["issue_price"].isna()
    | (df["issue_price"] <= 0)
    | df["target_open_gain_pct"].isna()
    | df["target_typical_gain_pct"].isna()
    | df["target_close_gain_pct"].isna()
]

print(f"Found {len(malformed_listing)} unlisted/malformed rows to drop:")
display_cols = ["company_name", "issue_type", "exchange", "open_date", "close_date", "listing_date", "issue_price", "target_close_gain_pct"]
print(malformed_listing[display_cols].to_string())

df = df.drop(index=malformed_listing.index).reset_index(drop=True)
print(f"\nRemaining listed equity IPOs: {len(df)}")

Found 4 unlisted/malformed rows to drop:
                        company_name   issue_type    exchange    open_date   close_date listing_date  issue_price  target_close_gain_pct
290  Trafiksol ITS Technologies Ltd.   Book_Built     BSE_SME  10-Sep-2024  12-Sep-2024          NaN         70.0                    NaN
749  Phytochem Remedies (India) Ltd.  Fixed_Price     BSE_SME  18-Dec-2025  22-Dec-2025          NaN         98.0                    NaN
867     Sri Priyanka Geo Commex Ltd.   Book_Built  NSE_EMERGE  24-Jun-2026  29-Jun-2026          NaN          NaN                    NaN
871              Twinkle Papers Ltd.   Book_Built     BSE_SME  29-Jun-2026  01-Jul-2026          NaN          NaN                    NaN

Remaining listed equity IPOs: 1060


---
## Step 3: Date Parsing & Chronological Consistency Check
Parse `open_date`, `close_date`, and `listing_date` into `datetime64`, verify `open_date <= close_date <= listing_date`, and sort chronologically by `listing_date` for clean time-series splits.

In [4]:
for dcol in ["open_date", "close_date", "listing_date"]:
    df[dcol] = pd.to_datetime(df[dcol], errors="coerce")
    print(f"{dcol:14s}: {df[dcol].isna().sum()} invalid/missing dates")

print(f"Anomalies where open_date > close_date   : {(df['open_date'] > df['close_date']).sum()}")
print(f"Anomalies where close_date > listing_date: {(df['close_date'] > df['listing_date']).sum()}")

df = df.sort_values(["listing_date", "close_date", "company_name"]).reset_index(drop=True)
print(f"Date range: {df['listing_date'].min().date()} to {df['listing_date'].max().date()}")

open_date     : 0 invalid/missing dates
close_date    : 0 invalid/missing dates
listing_date  : 0 invalid/missing dates
Anomalies where open_date > close_date   : 0
Anomalies where close_date > listing_date: 0
Date range: 2023-11-06 to 2026-10-15


---
## Step 4: Issue Basics & Bounds Audit (`price`, `size`, `holding`, `anchor`)
Audit issue-structure columns for out-of-bounds or structural zero values:
- `total_issue_size_cr <= 0` or `post_issue_mcap_cr <= 0`
- `promoter_hold_post_pct` and `tier1_anchor_share_pct` outside `[0, 100]`
- `fresh_issue_cr` / `ofs_size_cr` `NaN` values (pure OFS issues have `0.0` fresh issue; pure fresh issues have `0.0` OFS)

In [5]:
print("total_issue_size_cr <= 0 or NaN:", ((df["total_issue_size_cr"].isna()) | (df["total_issue_size_cr"] <= 0)).sum())
print("post_issue_mcap_cr  <= 0 or NaN:", ((df["post_issue_mcap_cr"].isna()) | (df["post_issue_mcap_cr"] <= 0)).sum())
print("promoter_hold_post_pct out of [0,100]:", ((df["promoter_hold_post_pct"] < 0) | (df["promoter_hold_post_pct"] > 100)).sum())
print("tier1_anchor_share_pct out of [0,100]:", ((df["tier1_anchor_share_pct"] < 0) | (df["tier1_anchor_share_pct"] > 100)).sum())

print(f"\nMissing fresh_issue_cr (pure OFS issues)  : {df['fresh_issue_cr'].isna().sum()}")
print(f"Missing ofs_size_cr    (pure fresh issues): {df['ofs_size_cr'].isna().sum()}")
print(f"Missing anchor_portion_cr                 : {df['anchor_portion_cr'].isna().sum()}")

# Fill structural zeros for issue components and anchor book
df["fresh_issue_cr"] = df["fresh_issue_cr"].fillna(0.0).clip(lower=0.0)
df["ofs_size_cr"] = df["ofs_size_cr"].fillna(0.0).clip(lower=0.0)
df["anchor_portion_cr"] = df["anchor_portion_cr"].fillna(0.0).clip(lower=0.0)
df["anchor_investor_count"] = df["anchor_investor_count"].fillna(0).clip(lower=0)
df["tier1_anchor_share_pct"] = df["tier1_anchor_share_pct"].fillna(0.0).clip(0.0, 100.0)
df["tier1_unique_houses"] = df["tier1_unique_houses"].fillna(0).clip(lower=0)

total_issue_size_cr <= 0 or NaN: 0
post_issue_mcap_cr  <= 0 or NaN: 0
promoter_hold_post_pct out of [0,100]: 0
tier1_anchor_share_pct out of [0,100]: 0

Missing fresh_issue_cr (pure OFS issues)  : 55
Missing ofs_size_cr    (pure fresh issues): 647
Missing anchor_portion_cr                 : 0


---
## Step 5: Financial Statements & Valuation Ratios Audit
Inspect financial columns (`rev_fy1..3`, `pat_fy1..3`, `ebitda_fy1..2`, `net_worth_fy1`, `borrowing_fy1`, `pe_post_ipo`, `price_to_book`, `roe_pct`, `roce_pct`) for malformed values before computing ratios:
1. **Negative `borrowing_fy1`:** Total debt cannot be negative (rounding artifact `-0.04` Cr on *Trust Fintech Ltd.*). Clipped at `0.0`.
2. **Non-positive `net_worth_fy1 <= 0`:** Companies with accumulated losses have negative pre-IPO net worth; `debt_to_equity` evaluates to `NaN` when `net_worth_fy1 <= 0`.
3. **Missing `rev_fy1`:** *Harshdeep Hortico Ltd.* has missing `rev_fy1` – margin and `price_to_sales` ratios safely evaluate to `NaN`.
4. **Extreme `roe_pct`, `roce_pct`, `price_to_book`, and `pe_post_ipo`:** Logged prior to domain-safe clipping.

In [6]:
print("borrowing_fy1 < 0:")
print(df[df["borrowing_fy1"] < 0][["company_name", "borrowing_fy1", "net_worth_fy1"]])

print("\nnet_worth_fy1 <= 0:")
print(df[df["net_worth_fy1"] <= 0][["company_name", "net_worth_fy1", "borrowing_fy1"]])

print("\nrev_fy1 <= 0 or NaN:")
print(df[(df["rev_fy1"].isna()) | (df["rev_fy1"] <= 0)][["company_name", "rev_fy1", "pat_fy1"]])

print(f"\nLoss-making companies (pat_fy1 <= 0): {(df['pat_fy1'] <= 0).sum()}")
print("Extreme roe_pct outside [-100, 150]:", df[(df["roe_pct"] < -100) | (df["roe_pct"] > 150)][["company_name", "roe_pct"]].to_dict("records"))
print("Extreme roce_pct outside [-100, 150]:", df[(df["roce_pct"] < -100) | (df["roce_pct"] > 150)][["company_name", "roce_pct"]].to_dict("records"))
print("Extreme price_to_book > 50:", df[df["price_to_book"] > 50][["company_name", "price_to_book"]].to_dict("records"))
print("Extreme pe_post_ipo > 150:", (df["pe_post_ipo"] > 150).sum(), "| <= 0:", (df["pe_post_ipo"] <= 0).sum())

# Fix negative borrowing artifact
df.loc[df["borrowing_fy1"] < 0, "borrowing_fy1"] = 0.0

borrowing_fy1 < 0:
           company_name  borrowing_fy1  net_worth_fy1
134  Trust Fintech Ltd.          -0.04          27.19

net_worth_fy1 <= 0:
               company_name  net_worth_fy1  borrowing_fy1
266   Premier Energies Ltd.          -0.24         763.54
533    Indiqube Spaces Ltd.          -3.11         343.96
695          Pine Labs Ltd.       -2244.27         829.49
947  Purple Style Labs Ltd.         -52.28         371.40

rev_fy1 <= 0 or NaN:
              company_name  rev_fy1  pat_fy1
76  Harshdeep Hortico Ltd.      NaN     -0.0

Loss-making companies (pat_fy1 <= 0): 31
Extreme roe_pct outside [-100, 150]: [{'company_name': 'Voler Car Ltd.', 'roe_pct': 249.01}, {'company_name': 'Ather Energy Ltd.', 'roe_pct': -194.0}, {'company_name': 'Meesho Ltd.', 'roe_pct': -252.37}]
Extreme roce_pct outside [-100, 150]: [{'company_name': 'Addictive Learning Technology Ltd.', 'roce_pct': 157.53}, {'company_name': 'Ather Energy Ltd.', 'roce_pct': -118.69}]
Extreme price_to_book > 50: [

---
## Step 6: Subscription, GMP & Macro Regime Audit
Inspect demand (`sub_*`, `app_*`), grey market (`gmp_*`), and macro regime columns:
- Verify `sub_total_x` has no missing values among listed IPOs.
- Preserve `NaN` vs `0.0` distinction for `sub_qib_*` (no QIB quota vs zero bids).
- Drop sparse `sub_bnii_x` and `sub_snii_x`, keeping universal `sub_nii_total_x`.
- Verify all 6 macro regime indicators (`nifty50_*`, `smallcap250_*`, `india_vix_*`) are 100% populated.

In [7]:
print("Missing sub_total_x:", df["sub_total_x"].isna().sum())
print("Missing sub_nii_total_x:", df["sub_nii_total_x"].isna().sum())
print("Missing sub_rii_x:", df["sub_rii_x"].isna().sum())
print("Missing gmp_close_day_inr:", df["gmp_close_day_inr"].isna().sum())

macro_cols = [
    "nifty50_ret_5d_pct", "nifty50_ret_22d_pct",
    "smallcap250_ret_5d_pct", "smallcap250_ret_22d_pct",
    "india_vix_close", "india_vix_5d_chg_pct"
]
print("\nMacro regime missing counts:")
print(df[macro_cols].isna().sum())

inconsistent_qib = df[df["sub_qib_total_x"].isna() & df["sub_qib_fii_x"].notna()]
print(f"\nInconsistent QIB rows (total NaN but sub-bucket present): {len(inconsistent_qib)}")

Missing sub_total_x: 0
Missing sub_nii_total_x: 105
Missing sub_rii_x: 22
Missing gmp_close_day_inr: 4

Macro regime missing counts:
nifty50_ret_5d_pct         0
nifty50_ret_22d_pct        0
smallcap250_ret_5d_pct     0
smallcap250_ret_22d_pct    0
india_vix_close            0
india_vix_5d_chg_pct       0
dtype: int64

Inconsistent QIB rows (total NaN but sub-bucket present): 0


---
## Step 7: Feature Engineering & Derived Ratios
Engineer scale-invariant financial ratios, GMP percentage & momentum features, broker consensus metrics, and structural indicator flags:
1. **Issue Composition:** `ofs_pct`
2. **Grey Market Sentiment:** `gmp_close_day_pct`, `gmp_momentum_pct` (computed from `gmp_close_day_pct - gmp_avg_pct`; collinear `gmp_avg_pct` is not retained)
3. **Financial Growth & Margins:** `rev_yoy_pct`, `pat_yoy_pct`, `rev_2yr_cagr`, `ebitda_yoy_pct`, `pat_margin_pct`, `ebitda_margin_pct`
4. **Leverage & Valuation:** `debt_to_equity`, `price_to_sales`, `price_to_book` (using post-issue net worth `net_worth_fy1 + fresh_issue_cr` as fallback), `roe_pct`, `roce_pct`
5. **Loss-Making & Relative P/E:** Compute `is_loss_making` **before** converting `pe_post_ipo <= 0` to `NaN`, then compute `relative_pe = pe_post_ipo / peer_avg_pe` (clipped at `10.0`)
6. **Broker Sentiment:** `broker_total_reviews`, `broker_sub_ratio`
7. **Binary Indicator Flags:** `has_listed_peers`, `has_multiyear_financials`, `has_qib_quota`, `has_anchor_book`, `gmp_available`

In [8]:
# 7.1 Issue composition
df["ofs_pct"] = np.where(df["total_issue_size_cr"] > 0, df["ofs_size_cr"] / df["total_issue_size_cr"], 0.0).clip(0.0, 1.0)

# 7.2 GMP percentage & momentum (relative to issue_price)
df["gmp_close_day_pct"] = (df["gmp_close_day_inr"] / df["issue_price"]) * 100.0
gmp_avg_pct = (df["gmp_avg_inr"] / df["issue_price"]) * 100.0
df["gmp_momentum_pct"] = df["gmp_close_day_pct"] - gmp_avg_pct

# 7.3 Financial growth & margins
df["rev_yoy_pct"] = np.where(df["rev_fy2"] > 0, ((df["rev_fy1"] - df["rev_fy2"]) / df["rev_fy2"]) * 100.0, np.nan)
df["rev_yoy_pct"] = df["rev_yoy_pct"].clip(-100.0, 500.0)

df["pat_yoy_pct"] = np.where(df["pat_fy2"] > 0, ((df["pat_fy1"] - df["pat_fy2"]) / df["pat_fy2"]) * 100.0, np.nan)
df["pat_yoy_pct"] = df["pat_yoy_pct"].clip(-200.0, 500.0)

df["rev_2yr_cagr"] = np.where(
    (df["rev_fy1"] > 0) & (df["rev_fy3"] > 0),
    (np.sqrt(df["rev_fy1"] / df["rev_fy3"]) - 1.0) * 100.0,
    np.nan
)
df["rev_2yr_cagr"] = df["rev_2yr_cagr"].clip(-100.0, 300.0)

df["ebitda_yoy_pct"] = np.where(df["ebitda_fy2"] > 0, ((df["ebitda_fy1"] - df["ebitda_fy2"]) / df["ebitda_fy2"]) * 100.0, np.nan)
df["ebitda_yoy_pct"] = df["ebitda_yoy_pct"].clip(-100.0, 500.0)

df["pat_margin_pct"] = np.where(df["rev_fy1"] > 0, (df["pat_fy1"] / df["rev_fy1"]) * 100.0, np.nan)
df["pat_margin_pct"] = df["pat_margin_pct"].clip(-100.0, 100.0)

df["ebitda_margin_pct"] = np.where(df["rev_fy1"] > 0, (df["ebitda_fy1"] / df["rev_fy1"]) * 100.0, np.nan)
df["ebitda_margin_pct"] = df["ebitda_margin_pct"].clip(-100.0, 100.0)

# 7.4 Leverage & valuation
df["debt_to_equity"] = np.where(df["net_worth_fy1"] > 0, df["borrowing_fy1"] / df["net_worth_fy1"], np.nan)
df["debt_to_equity"] = df["debt_to_equity"].clip(0.0, 20.0)

df["price_to_sales"] = np.where(df["rev_fy1"] > 0, df["post_issue_mcap_cr"] / df["rev_fy1"], np.nan)
df["price_to_sales"] = df["price_to_sales"].clip(0.0, 50.0)

book_val_post = df["net_worth_fy1"] + df["fresh_issue_cr"]
derived_pb = np.where(book_val_post > 0, df["post_issue_mcap_cr"] / book_val_post, np.nan)
df["price_to_book"] = df["price_to_book"].fillna(pd.Series(derived_pb, index=df.index)).clip(0.0, 50.0)

df["roe_pct"] = df["roe_pct"].clip(-100.0, 150.0)
df["roce_pct"] = df["roce_pct"].clip(-100.0, 150.0)

# 7.5 Loss-making flag BEFORE neutralizing negative P/E
df["is_loss_making"] = np.where(
    df["pat_fy1"].notnull(),
    (df["pat_fy1"] <= 0).astype(float),
    np.where(df["pe_post_ipo"] < 0, 1.0, np.where(df["pe_post_ipo"] > 0, 0.0, np.nan))
)

df.loc[df["pe_post_ipo"] <= 0, "pe_post_ipo"] = np.nan
df["pe_post_ipo"] = df["pe_post_ipo"].clip(upper=150.0)

valid_pe = (df["pe_post_ipo"] > 0) & (df["peer_avg_pe"] > 0)
df["relative_pe"] = np.where(valid_pe, df["pe_post_ipo"] / df["peer_avg_pe"], np.nan)
df["relative_pe"] = df["relative_pe"].clip(upper=10.0)

# 7.6 Broker consensus
broker_cols = ["broker_sub", "broker_may", "broker_neutral", "broker_avoid"]
df["broker_total_reviews"] = df[broker_cols].fillna(0).sum(axis=1)
df["broker_sub_ratio"] = np.where(
    df["broker_total_reviews"] > 0,
    df["broker_sub"].fillna(0) / df["broker_total_reviews"],
    np.nan
)

# 7.7 Structural binary flags
df["has_listed_peers"] = df["peer_avg_pe"].notnull().astype(int)
df["has_multiyear_financials"] = df["rev_fy2"].notnull().astype(int)
missing_cat_breakdown = df["sub_qib_total_x"].isna() & df["sub_nii_total_x"].isna() & df["sub_rii_x"].isna()
df["has_qib_quota"] = np.where(missing_cat_breakdown, np.nan, df["sub_qib_total_x"].notnull().astype(float))
df["has_anchor_book"] = (df["anchor_portion_cr"] > 0).astype(int)
df["gmp_available"] = df["gmp_close_day_pct"].notnull().astype(int)

print("Engineered derived ratios and flags successfully.")

Engineered derived ratios and flags successfully.


---
## Step 8: High-Cardinality Categorical Normalization (`sector` & `lead_manager`)
Reduce fragmentation on ~1,060 rows:
1. **`sector`:** Map 153 raw `industry` labels into 14 standardized macro sector buckets.
2. **`lead_manager_grouped`:** Normalize whitespace and group lead managers with historical count `< 5` into `"Other"`.

In [9]:
def map_industry_to_sector(ind):
    if pd.isna(ind) or not str(ind).strip():
        return "Other/Diversified"
    s = str(ind).lower()
    if any(k in s for k in ["bank", "nbfc", "financial", "fintech", "insurance", "stockbroking", "asset management", "microfinance", "capital market", "finance"]):
        return "Financials"
    if any(k in s for k in ["pharma", "hospital", "healthcare", "medical", "diagnostic", "biotech", "path lab"]):
        return "Healthcare & Pharma"
    if any(k in s for k in ["software", "it ", "it-", "information technology", "e-learning", "bpo", "kpo", "computers", "cyber", "saas", "telecom"]):
        return "IT & Tech"
    if any(k in s for k in ["auto", "wheeler", "tractor", "passenger car", "commercial vehicle"]):
        return "Auto & Ancillaries"
    if any(k in s for k in ["chemical", "petrochemical", "pesticide", "agrochemical", "fertilizer", "dyes", "pigment"]):
        return "Chemicals & Petrochemicals"
    if any(k in s for k in ["real estate", "residential", "commercial project", "housing", "reit"]):
        return "Real Estate"
    if any(k in s for k in ["construction", "civil", "infrastructure", "road asset", "cement", "granite", "marble", "building material", "sanitary", "ceramics", "water supply"]):
        return "Infrastructure & Construction"
    if any(k in s for k in ["logistics", "transport", "shipping", "airline", "courier", "supply chain", "port", "warehousing"]):
        return "Logistics & Transport"
    if any(k in s for k in ["power", "energy", "solar", "oil", "gas", "lpg", "cng", "coal", "petroleum", "utility", "utilities"]):
        return "Energy & Utilities"
    if any(k in s for k in ["media", "entertainment", "advertising", "film", "printing", "publication", "event", "exhibitions"]):
        return "Media & Entertainment"
    if any(k in s for k in ["agricult", "seed", "dairy", "seafood", "tea", "coffee", "plantation"]):
        return "Agriculture"
    if any(k in s for k in ["garment", "apparel", "textile", "jewellery", "gems", "watch", "food", "fmcg", "retail", "e-retail", "e-commerce", "furniture", "houseware", "footwear", "personal care", "hotel", "restaurant", "tour", "travel", "leisure", "appliance", "breweries", "consumer"]):
        return "Consumer Goods & Retail"
    if any(k in s for k in ["industrial", "electrical", "steel", "metal", "aluminium", "copper", "zinc", "castings", "forging", "cable", "packaging", "paper", "plastic", "aerospace", "defense", "defence", "engineering", "glass", "rubber", "waste", "compressor", "pump", "machinery"]):
        return "Industrials & Capital Goods"
    return "Other/Diversified"

df["sector"] = df["industry"].apply(map_industry_to_sector)
print("Mapped 14 Macro Sectors:")
print(df["sector"].value_counts())

# Group rare lead managers (< 5 IPOs -> "Other")
lm_clean = df["lead_manager"].fillna("Other").astype(str).str.strip().replace("", "Other")
lm_counts = lm_clean.value_counts()
frequent_lms = set(lm_counts[lm_counts >= 5].index)
df["lead_manager_grouped"] = np.where(lm_clean.isin(frequent_lms), lm_clean, "Other")
print(f"\nLead Managers reduced from {lm_clean.nunique()} unique to {df['lead_manager_grouped'].nunique()} groups (threshold >= 5).")

Mapped 14 Macro Sectors:
sector
Industrials & Capital Goods      263
Consumer Goods & Retail          212
Other/Diversified                 94
IT & Tech                         84
Infrastructure & Construction     78
Healthcare & Pharma               69
Financials                        48
Logistics & Transport             46
Chemicals & Petrochemicals        45
Auto & Ancillaries                41
Agriculture                       27
Media & Entertainment             25
Energy & Utilities                16
Real Estate                       12
Name: count, dtype: int64

Lead Managers reduced from 124 unique to 63 groups (threshold >= 5).


---
## Step 9: `log1p` Transformations & SME Listing-Day Circuit Clipping
1. **`np.log1p` Transformation:** Applied in-place (after `.clip(lower=0)`) to heavily right-skewed issue size, market cap, anchor, and subscription columns while preserving `NaN`s.
2. **SME Target Circuit Clipping:** Clip SME Day-1 open gains to `[-20%, +90%]` and typical/close gains to `[-24%, +99.5%]` to match exchange listing-day bands.

In [10]:
log1p_cols = [
    "total_issue_size_cr", "post_issue_mcap_cr", "anchor_portion_cr",
    "sub_total_x", "sub_rii_x", "sub_nii_total_x", "sub_qib_total_x",
    "sub_qib_fii_x", "sub_qib_dfi_x", "sub_qib_mf_x", "sub_qib_other_x",
    "app_nii_x", "app_rii_x", "app_qib_fii", "app_qib_dfi", "app_qib_mf", "app_qib_other"
]

for col in log1p_cols:
    df[col] = np.log1p(df[col].clip(lower=0))

# SME circuit limit clipping
sme_mask = df["is_sme"] == 1
n_open_clip = ((df.loc[sme_mask, "target_open_gain_pct"] < -20.0) | (df.loc[sme_mask, "target_open_gain_pct"] > 90.0)).sum()
n_close_clip = ((df.loc[sme_mask, "target_close_gain_pct"] < -24.0) | (df.loc[sme_mask, "target_close_gain_pct"] > 99.5)).sum()

df.loc[sme_mask, "target_open_gain_pct"] = df.loc[sme_mask, "target_open_gain_pct"].clip(-20.0, 90.0)
df.loc[sme_mask, "target_typical_gain_pct"] = df.loc[sme_mask, "target_typical_gain_pct"].clip(-24.0, 99.5)
df.loc[sme_mask, "target_close_gain_pct"] = df.loc[sme_mask, "target_close_gain_pct"].clip(-24.0, 99.5)

print(f"Applied log1p to {len(log1p_cols)} skewed columns.")
print(f"Clipped SME targets -> open: {n_open_clip} rows, close: {n_close_clip} rows.")

Applied log1p to 17 skewed columns.
Clipped SME targets -> open: 43 rows, close: 39 rows.


---
## Step 10: Drop Redundant / Leakage Columns & Export Baseline Cleaned Dataset
Drop raw ₹ Cr financial sums, redundant component/price/sparse columns, unmapped high-cardinality strings, and Day-1 execution prices (leakage).

Identifier & date columns (`isin`, `symbol`, `bse_code`, `company_name`, `open_date`, `close_date`, `listing_date`) are retained strictly for chronological splitting and error auditing and will be excluded from `X` during modeling.

In [11]:
cols_to_drop = [
    # Raw financial sums
    "rev_fy1", "rev_fy2", "rev_fy3",
    "pat_fy1", "pat_fy2", "pat_fy3",
    "ebitda_fy1", "ebitda_fy2",
    "net_worth_fy1", "borrowing_fy1",
    # Redundant raw components & prices
    "fresh_issue_cr", "ofs_size_cr", "price_band_high", "issue_price",
    "gmp_close_day_inr", "gmp_avg_inr",
    "broker_sub", "broker_may", "broker_neutral", "broker_avoid",
    "sub_bnii_x", "sub_snii_x", "anchor_investor_count", "peer_avg_pe",
    # Raw unmapped strings
    "industry", "lead_manager",
    # Day-1 execution price leakage
    "list_open", "list_high", "list_low", "list_close", "list_typical_price",
]

df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns]).copy()

# Round floats to model-useful precision (4 decimals for log1p/fractions, 2 decimals for pct/ratios)
four_dec_cols = [c for c in log1p_cols + ["ofs_pct", "broker_sub_ratio"] if c in df_clean.columns]
two_dec_cols = [c for c in df_clean.select_dtypes(include=[np.number]).columns if c not in four_dec_cols and df_clean[c].dtype != "int64"]
df_clean[four_dec_cols] = df_clean[four_dec_cols].round(4)
df_clean[two_dec_cols] = df_clean[two_dec_cols].round(2)

num_cols = df_clean.select_dtypes(include=[np.number]).columns
inf_count = np.isinf(df_clean[num_cols]).sum().sum()
print(f"Infinite values check: {inf_count}")
print(f"Final cleaned dataset shape: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")

df_clean.to_csv(CLEAN_PATH, index=False, encoding="utf-8-sig")
print(f"Exported baseline cleaned dataset to: {CLEAN_PATH}")
print("\nFinal Columns:")
for i, c in enumerate(df_clean.columns, 1):
    print(f"  {i:2d}. {c:28s} | dtype={str(df_clean[c].dtype):10s} | nulls={df_clean[c].isna().sum()}")

Infinite values check: 0
Final cleaned dataset shape: 1060 rows, 66 columns
Exported baseline cleaned dataset to: ipo_listing_gain_clean.csv

Final Columns:
   1. isin                         | dtype=str        | nulls=0
   2. symbol                       | dtype=str        | nulls=0
   3. bse_code                     | dtype=str        | nulls=381
   4. company_name                 | dtype=str        | nulls=0
   5. is_sme                       | dtype=int64      | nulls=0
   6. exchange                     | dtype=str        | nulls=0
   7. issue_type                   | dtype=str        | nulls=0
   8. open_date                    | dtype=datetime64[us] | nulls=0
   9. close_date                   | dtype=datetime64[us] | nulls=0
  10. listing_date                 | dtype=datetime64[us] | nulls=0
  11. priced_at_upper_band         | dtype=int64      | nulls=0
  12. total_issue_size_cr          | dtype=float64    | nulls=0
  13. post_issue_mcap_cr           | dtype=float64    | nulls